# 07 — Target-verifier case mining (#91)

**Objective:** mine real 2025-26 comments into labeling queues for the
sentiment-target verifier's eval suite (`tests/eval/target_cases.yaml`). The
verifier answers, for a polar comment, *toward whom is this sentiment
directed?* — a player or null. Its failure families are **misdirected
sentiment**: the pass-1 label is right about the comment's tone but the tone is
not aimed at the attributed player (wrong-player target, non-player target,
subject-not-target). Two control families keep an always-null verifier from
scoring perfectly: **true_toward** (the sentiment does land on the attributed
player) and **readmit_affirm** (a gate-dropped NULL-`sentiment_player` row the
verifier must recover).

**Deliverable:** three queue YAMLs under `data/2025-26/reference/eval_mining/`
(gitignored; regenerated byte-identically from the frozen inputs), one per
source stratum:

| Queue | Source | What it feeds |
|---|---|---|
| `candidates_target_receipts.yaml` | shipped polar receipts (`comment_samples.parquet`) | the three failure families + `true_toward` |
| `candidates_target_nullp.yaml` | pre-gate polar NULL-`p` top-K, re-derived from `sentiment.parquet` | `readmit_affirm` + the class-1 misfires |
| `candidates_target_random.yaml` | random polar named-`p` and NULL-`p` strata | prevalence prior for the candidate-pool depth K |

Candidates carry `proposed_target` / `proposed_category` (both null —
deliberately unanchored); the labeler fills and renames them. **No prompt
change happens in this notebook or its PR.**

## Load Data

Guardrails (05 house pattern):

- **`body` is projected in exactly one place**: the §3 fetch, restricted to the
  ids §2 selected. The attribution frame (§1) is body-free; `comment_samples`
  already carries its bodies. Queues cap at the per-stratum quotas.
- Every derived artifact is guarded on existence with a staleness warning if it
  predates its source.
- **Determinism:** random strata sample by `ORDER BY md5(id)`; ranked strata
  order by `(rank, score desc, comment_id)`. Never `random()` or a seed.
- Queues carry `proposed_*`, not `expected_target` / `category`: pasting an
  unlabeled candidate into `target_cases.yaml` fails `load_target_cases()`
  loudly (missing keys).

In [1]:
import hashlib
import os
import sys
from pathlib import Path

# Bootstrap: run from anywhere (walk up to the repo root, chdir).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import duckdb
import polars as pl
import yaml

from pipeline.aggregation import resolve_player
from pipeline.evaluation import load_cases
from pipeline.targets import TARGET_CATEGORIES, load_target_cases
from utils.constants import COMMENT_SAMPLES_MAX_BODY_CHARS, COMMENT_SAMPLES_MIN_CONFIDENCE
from utils.paths import get_data_dir
from utils.player_config import build_alias_to_player_map
from utils.season_config import get_active_season

SEASON = "2025-26"  # pinned: this notebook is season-scoped (notebooks/2025-26/)
assert get_active_season() == SEASON, "active season flipped — review before re-running"

SENTIMENT = get_data_dir(season=SEASON) / "processed" / "sentiment.parquet"
SAMPLES = get_data_dir(season=SEASON) / "dashboard" / "comment_samples.parquet"
PLAYER_OVERALL = get_data_dir(season=SEASON) / "dashboard" / "player_overall.parquet"
MINING_DIR = get_data_dir(season=SEASON) / "reference" / "eval_mining"
MINING_DIR.mkdir(parents=True, exist_ok=True)

for p in (SENTIMENT, SAMPLES, PLAYER_OVERALL):
    assert p.exists(), f"missing input: {p}"

POLAR = ["pos", "neg"]
QUALIFIED_MIN_COMMENTS = 5000  # the official-ranking volume bar
STORYLINE = ["Luka Doncic", "Anthony Davis"]  # the 08-16 evidence cells, taken whole

# Dedup targets: both suites' texts/ids (a receipt may already be a sentiment case).
_sent_cases = load_cases()
_target_cases = load_target_cases()
SEEN_TEXTS = {c.text.strip().lower() for c in [*_sent_cases, *_target_cases]}
SEEN_IDS = {c.comment_id for c in [*_sent_cases, *_target_cases] if c.comment_id}


def warn_if_stale(pq: Path, raw: Path) -> None:
    """Companion to the existence-only reuse guards: flag a persisted
    artifact older than the file it derives from."""
    if pq.exists() and pq.stat().st_mtime < raw.stat().st_mtime:
        print(f"WARNING: {pq.name} predates {raw.name} — delete it to force a rescan")


def md5_key(s: str) -> str:
    return hashlib.md5(s.encode()).hexdigest()


def dedup_take(rows: list[dict], n: int) -> list[dict]:
    """First n rows (already ordered) not colliding with existing cases or
    earlier queues; body hygiene applied here so every stratum shares it.
    Mutates the shared SEEN sets, so cross-queue dedup falls out of order."""
    out: list[dict] = []
    for r in rows:
        body = r["body"]
        if body is None or body in ("[deleted]", "[removed]"):
            continue
        if len(body) > COMMENT_SAMPLES_MAX_BODY_CHARS:
            continue
        key = body.strip().lower()
        if key in SEEN_TEXTS or r["comment_id"] in SEEN_IDS:
            continue
        SEEN_TEXTS.add(key)
        SEEN_IDS.add(r["comment_id"])
        out.append(r)
        if len(out) == n:
            break
    return out


def make_candidate(cid: str, r: dict, note: str, context: dict) -> dict:
    """Queue entry in the documented field order. Unanchored: the labeler
    decides target and category."""
    return {
        "id": cid,
        "text": r["body"].strip(),
        "sentiment": r["sentiment"],
        "attributed_player": r["attributed_player"],
        "proposed_target": None,
        "proposed_category": None,
        "source": "mined-v2",
        "comment_id": r["comment_id"],
        "note": note,
        "context": context,
    }


def queue_header(filename: str, section: str) -> str:
    cats = " | ".join(TARGET_CATEGORIES)
    return (
        f"# {filename} — written by notebooks/2025-26/07_target_case_mining.ipynb ({section})\n"
        "#\n"
        "# LABELING (the verifier never sees attributed_player; you judge against it):\n"
        "#   1. Set proposed_target: the canonical player the comment's SENTIMENT is\n"
        "#      directed at, or null. A player is the target only if identifiable from\n"
        "#      the text itself (named, or unambiguously described); an unnamed 'him'\n"
        "#      is null — the verifier must not guess. A named player is the target if\n"
        "#      ANY of the sentiment lands on them, even when a decision-maker is also\n"
        "#      mocked. Untracked people (a GM, a referee, a retired player) are null.\n"
        f"#   2. Set proposed_category, one of: {cats}\n"
        "#        wrong_player       -> a different tracked player, identifiable in-text\n"
        "#        non_player         -> target is not a tracked player (front office, coach,\n"
        "#                              refs, media, fans, untracked/retired player) -> null\n"
        "#        subject_not_target -> attributed player is in the scene as victim, foil,\n"
        "#                              standard, beneficiary or bystander, not target -> null\n"
        "#        true_toward        -> target == attributed_player (classifier named them)\n"
        "#        readmit_affirm     -> target == attributed_player on a NULL-p row\n"
        "#   3. REJECT (delete the entry, note the id) when:\n"
        "#        - you would relabel the pass-1 sentiment (sarcasm, banter, neutral report):\n"
        "#          the target question is undefined; it is a cases.yaml candidate instead\n"
        "#        - the target is undecidable from the body alone (needs the parent/thread)\n"
        "#        - the mention is an alias-identity false positive (a referee named Davis)\n"
        "#   4. RENAME proposed_target -> expected_target, proposed_category -> category;\n"
        "#      delete the `context:` block from keepers (fold anything worth keeping into\n"
        "#      `note`).\n"
        "#   5. Copy survivors into tests/eval/target_cases.yaml; every used category\n"
        "#      needs a meta.category_floors entry (placeholder 0.0).\n"
        "#   Unrenamed or category-inconsistent entries fail load_target_cases() by design.\n"
        "#\n"
    )


def write_queue(path: Path, rows: list[dict], section: str) -> None:
    body = yaml.safe_dump(
        {"candidates": rows}, sort_keys=False, allow_unicode=True, width=4096
    )
    path.write_text(queue_header(path.name, section) + body)
    print(f"wrote {len(rows)} candidates -> {path}")


duckdb.sql("SET temp_directory = '/tmp/duckdb_mining_spill'")

print(f"sentiment.parquet: {SENTIMENT}  ({SENTIMENT.stat().st_size / 1e6:.0f} MB)")
print(f"comment_samples:   {SAMPLES}")
print(f"Existing cases: {len(_sent_cases)} sentiment, {len(_target_cases)} target  (dedup sets primed)")

sentiment.parquet: data/2025-26/processed/sentiment.parquet  (177 MB)
comment_samples:   data/2025-26/dashboard/comment_samples.parquet
Existing cases: 99 sentiment, 0 target  (dedup sets primed)


## 1. Polar attribution frame (body-free)

One pass over `sentiment.parquet` *without* `body`: polar rows only, attributed
exactly as aggregation does (`resolve_player` over `mentioned_players` +
`sentiment_player` under the active `players.yaml`). Persisted as
`v2_polar_attributed.parquet`; every stratum below selects from it. The
NULL-`p` share here is the class the #86 gate removes from receipts.

In [2]:
# §1 — the attribution frame. Guarded; delete the parquet to force a rebuild.
ATTR_PQ = MINING_DIR / "v2_polar_attributed.parquet"
warn_if_stale(ATTR_PQ, SENTIMENT)

if not ATTR_PQ.exists():
    alias_map = build_alias_to_player_map()
    frame = (
        pl.scan_parquet(SENTIMENT)
        .select(
            "comment_id", "sentiment", "confidence", "sentiment_player",
            "mentioned_players", "score", "author_flair_text",
        )
        .filter(pl.col("sentiment").is_in(POLAR))
        .collect()
    )
    frame = (
        frame.with_columns(
            pl.struct(["mentioned_players", "sentiment_player"])
            .map_elements(
                lambda row: resolve_player(
                    row["mentioned_players"], row["sentiment_player"], alias_map
                ),
                return_dtype=pl.Utf8,
            )
            .alias("attributed_player"),
            pl.col("mentioned_players").list.len().cast(pl.Int64).alias("n_players"),
        )
        .filter(pl.col("attributed_player").is_not_null())
        .drop("mentioned_players")
    )
    frame.write_parquet(ATTR_PQ)

attr = pl.read_parquet(ATTR_PQ)
qualified = set(
    pl.read_parquet(PLAYER_OVERALL)
    .filter(pl.col("comment_count") >= QUALIFIED_MIN_COMMENTS)["attributed_player"]
)
assert set(STORYLINE) <= qualified

print(f"polar attributed rows: {attr.height:,}")
print(attr.group_by(pl.col("sentiment_player").is_null().alias("null_p"))
      .agg(pl.len().alias("rows"))
      .with_columns((pl.col("rows") / attr.height).round(3).alias("share"))
      .sort("null_p"))
print(f"qualified players (>= {QUALIFIED_MIN_COMMENTS:,} comments): {len(qualified)}")

polar attributed rows: 1,119,839
shape: (2, 3)
┌────────┬─────────┬───────┐
│ null_p ┆ rows    ┆ share │
│ ---    ┆ ---     ┆ ---   │
│ bool   ┆ u32     ┆ f64   │
╞════════╪═════════╪═══════╡
│ false  ┆ 1009995 ┆ 0.902 │
│ true   ┆ 109844  ┆ 0.098 │
└────────┴─────────┴───────┘
qualified players (>= 5,000 comments): 67


## 2. Candidate selection (ids only)

Three strata, selected from §1 and `comment_samples` without touching bodies
(the shipped receipts already carry theirs). Over-fetch 2–3× where a body
filter or dedup follows.

- **A. Shipped receipts** — the storyline players' full polar cells (the 08-16
  evidence: Luka 10/10, AD 6/10), plus ranks 1–2 negative for ten other
  qualified players and rank-1 positive for ten more (`md5(player)` order).
- **B. Pre-gate NULL-`p` top-K** — rank every gate-eligible polar row (confidence
  ≥ floor) by score within its `(attributed_player, sentiment)` cell exactly as
  `build_comment_samples` would *without* the target gate, then keep the
  NULL-`p` rows at the top of qualified cells. The rank-1 negatives here are
  the 08-15 "16 of 67" misfire list.
- **C. Random strata** — `md5(comment_id)`-ordered polar rows at or above the
  confidence floor: named-`p` (class-2 prevalence at the aggregate level) and
  NULL-`p` (the unbiased class-1 split).

In [3]:
# §2A — shipped receipts. comment_samples has bodies; join §1 for classifier context.
samples = (
    pl.read_parquet(SAMPLES)
    .filter(pl.col("sentiment").is_in(POLAR))
    .join(
        attr.select("comment_id", "confidence", "sentiment_player", "n_players"),
        on="comment_id",
        how="left",
    )
)
others = sorted(qualified - set(STORYLINE), key=md5_key)
NEG_PLAYERS, POS_PLAYERS = others[:10], others[10:20]

a_storyline = samples.filter(pl.col("attributed_player").is_in(STORYLINE))
a_neg = samples.filter(
    pl.col("attributed_player").is_in(NEG_PLAYERS)
    & (pl.col("sentiment") == "neg")
    & (pl.col("rank") <= 2)
)
a_pos = samples.filter(
    pl.col("attributed_player").is_in(POS_PLAYERS)
    & (pl.col("sentiment") == "pos")
    & (pl.col("rank") == 1)
)
receipts_sel = (
    pl.concat([a_storyline, a_neg, a_pos])
    .sort(["attributed_player", "sentiment", "rank"])
)
print(f"A: storyline {a_storyline.height} + neg {a_neg.height} + pos {a_pos.height} "
      f"= {receipts_sel.height} rows")
print("neg players:", NEG_PLAYERS)
print("pos players:", POS_PLAYERS)

A: storyline 40 + neg 20 + pos 10 = 70 rows
neg players: ['Josh Hart', 'Isaiah Hartenstein', 'Evan Mobley', 'Chet Holmgren', "De'Aaron Fox", 'Deni Avdija', 'Bam Adebayo', 'Alex Caruso', 'Dillon Brooks', 'LeBron James']
pos players: ['Jamal Murray', 'Desmond Bane', 'Tyrese Maxey', 'Joel Embiid', 'Jimmy Butler', 'Zion Williamson', 'Stephen Curry', 'Klay Thompson', 'Jalen Duren', 'Donovan Mitchell']


In [4]:
# §2B — pre-gate ranking: build_comment_samples' candidacy minus the target gate
# (the body cap is applied after the fetch, so ranks are approximate at the margin).
pregate = (
    attr.filter(
        (pl.col("confidence") >= COMMENT_SAMPLES_MIN_CONFIDENCE)
        & pl.col("attributed_player").is_in(qualified)
    )
    .sort(["score", "confidence", "comment_id"], descending=[True, True, False], nulls_last=True)
    .with_columns(
        (pl.int_range(pl.len()).over(["attributed_player", "sentiment"]) + 1).alias("pregate_rank")
    )
)
nullp_top = pregate.filter(
    pl.col("sentiment_player").is_null()
    & (
        ((pl.col("sentiment") == "neg") & (pl.col("pregate_rank") <= 3))
        | ((pl.col("sentiment") == "pos") & (pl.col("pregate_rank") <= 2))
    )
).sort(["pregate_rank", "score", "comment_id"], descending=[False, True, False])

rank1_neg = nullp_top.filter((pl.col("sentiment") == "neg") & (pl.col("pregate_rank") == 1))
print(f"B: {nullp_top.height} NULL-p rows in the top of qualified cells; "
      f"{rank1_neg.height} are rank-1 negatives (the 08-15 count was 16)")
print(rank1_neg.select("attributed_player", "score").head(20))

B: 71 NULL-p rows in the top of qualified cells; 16 are rank-1 negatives (the 08-15 count was 16)
shape: (16, 2)
┌───────────────────┬───────┐
│ attributed_player ┆ score │
│ ---               ┆ ---   │
│ str               ┆ i64   │
╞═══════════════════╪═══════╡
│ Nikola Jokic      ┆ 4098  │
│ Desmond Bane      ┆ 3040  │
│ Paul George       ┆ 2789  │
│ Austin Reaves     ┆ 2406  │
│ Tyrese Maxey      ┆ 1506  │
│ …                 ┆ …     │
│ Jared McCain      ┆ 352   │
│ OG Anunoby        ┆ 206   │
│ Aaron Gordon      ┆ 185   │
│ Kon Knueppel      ┆ 134   │
│ Dylan Harper      ┆ 102   │
└───────────────────┴───────┘


In [5]:
# §2C — random strata, md5(comment_id) order. Over-fetch 3x for the body filter.
N_RANDOM = 15
eligible = attr.filter(pl.col("confidence") >= COMMENT_SAMPLES_MIN_CONFIDENCE).with_columns(
    pl.col("comment_id").map_elements(md5_key, return_dtype=pl.Utf8).alias("md5")
)
random_named = eligible.filter(pl.col("sentiment_player").is_not_null()).sort("md5").head(N_RANDOM * 3)
random_nullp = eligible.filter(pl.col("sentiment_player").is_null()).sort("md5").head(N_RANDOM * 3)
print(f"C: {random_named.height} named-p + {random_nullp.height} NULL-p over-fetched")

C: 45 named-p + 45 NULL-p over-fetched


## 3. The one body pass

Fetch bodies for the §2B/§2C ids only (`comment_samples` rows already have
theirs). One filtered projection of `sentiment.parquet`.

In [6]:
# §3 — bodies for the selected ids.
fetch_ids = sorted(
    set(nullp_top["comment_id"]) | set(random_named["comment_id"]) | set(random_nullp["comment_id"])
)
_id_list = ", ".join(f"'{i}'" for i in fetch_ids)
bodies = duckdb.sql(f"""
    SELECT comment_id, body
    FROM read_parquet('{SENTIMENT}')
    WHERE comment_id IN ({_id_list})
""").pl()
assert bodies.height == len(fetch_ids), "id -> body fetch is not 1:1"
print(f"fetched {bodies.height} bodies")

fetched 161 bodies


## 4. Queues

Assemble and write the three queues. Quotas are per stratum; `dedup_take`
applies body hygiene, the receipts body cap, and cross-suite/cross-queue dedup
in order — so the receipts queue (the evidence cells) claims a comment before
the strata can.

In [7]:
# §4a — receipts queue (A).
receipt_rows = dedup_take(receipts_sel.to_dicts(), receipts_sel.height)
receipts_queue = [
    make_candidate(
        cid=f"receipt-m{i:02d}",
        r=r,
        note=f"shipped {r['sentiment']} receipt, rank {r['rank']}; "
        "classifier named a target — judge whether the sentiment lands on the attributed player",
        context={
            "rank": r["rank"],
            "score": r["score"],
            "confidence": r["confidence"],
            "sentiment_player_raw": r["sentiment_player"],
            "n_players": r["n_players"],
            "fan_team": r["fan_team"],
        },
    )
    for i, r in enumerate(receipt_rows, start=1)
]
write_queue(MINING_DIR / "candidates_target_receipts.yaml", receipts_queue, "§4a")
print(pl.DataFrame(receipt_rows).group_by("attributed_player", "sentiment").len().sort("attributed_player", "sentiment"))

wrote 70 candidates -> data/2025-26/reference/eval_mining/candidates_target_receipts.yaml
shape: (24, 3)
┌───────────────────┬───────────┬─────┐
│ attributed_player ┆ sentiment ┆ len │
│ ---               ┆ ---       ┆ --- │
│ str               ┆ str       ┆ u32 │
╞═══════════════════╪═══════════╪═════╡
│ Alex Caruso       ┆ neg       ┆ 2   │
│ Anthony Davis     ┆ neg       ┆ 10  │
│ Anthony Davis     ┆ pos       ┆ 10  │
│ Bam Adebayo       ┆ neg       ┆ 2   │
│ Chet Holmgren     ┆ neg       ┆ 2   │
│ …                 ┆ …         ┆ …   │
│ Luka Doncic       ┆ neg       ┆ 10  │
│ Luka Doncic       ┆ pos       ┆ 10  │
│ Stephen Curry     ┆ pos       ┆ 1   │
│ Tyrese Maxey      ┆ pos       ┆ 1   │
│ Zion Williamson   ┆ pos       ┆ 1   │
└───────────────────┴───────────┴─────┘


In [8]:
# §4b — pre-gate NULL-p queue (B): rank-1 negatives first, then deeper ranks.
NULLP_NEG, NULLP_POS = 25, 10
_nullp_rows = nullp_top.join(bodies, on="comment_id", how="left").to_dicts()
nullp_rows = dedup_take([r for r in _nullp_rows if r["sentiment"] == "neg"], NULLP_NEG) + dedup_take(
    [r for r in _nullp_rows if r["sentiment"] == "pos"], NULLP_POS
)
nullp_queue = [
    make_candidate(
        cid=f"nullp-m{i:02d}",
        r=r,
        note=f"pre-gate {r['sentiment']} rank {r['pregate_rank']}; classifier declined a target "
        "(gate-dropped) — readmit_affirm if the sentiment lands on the attributed player",
        context={
            "pregate_rank": r["pregate_rank"],
            "score": r["score"],
            "confidence": r["confidence"],
            "n_players": r["n_players"],
            "author_flair_text": r["author_flair_text"],
        },
    )
    for i, r in enumerate(nullp_rows, start=1)
]
write_queue(MINING_DIR / "candidates_target_nullp.yaml", nullp_queue, "§4b")

wrote 35 candidates -> data/2025-26/reference/eval_mining/candidates_target_nullp.yaml


In [9]:
# §4c — random strata queue (C).
_named = dedup_take(random_named.join(bodies, on="comment_id", how="left").to_dicts(), N_RANDOM)
_nullp = dedup_take(random_nullp.join(bodies, on="comment_id", how="left").to_dicts(), N_RANDOM)
random_queue = [
    make_candidate(
        cid=f"random-m{i:02d}",
        r=r,
        note=("random polar named-p row (class-2 prevalence stratum)"
              if r["sentiment_player"] is not None
              else "random polar NULL-p row (class-1 stratum; readmit_affirm if it lands on the attributed player)"),
        context={
            "stratum": "named_p" if r["sentiment_player"] is not None else "null_p",
            "score": r["score"],
            "confidence": r["confidence"],
            "sentiment_player_raw": r["sentiment_player"],
            "n_players": r["n_players"],
            "author_flair_text": r["author_flair_text"],
        },
    )
    for i, r in enumerate([*_named, *_nullp], start=1)
]
write_queue(MINING_DIR / "candidates_target_random.yaml", random_queue, "§4c")

wrote 30 candidates -> data/2025-26/reference/eval_mining/candidates_target_random.yaml


## 5. Verdict & handoff

Queue inventory, determinism hashes (re-running against the frozen inputs must
reproduce these), and collision re-checks against both suites.

In [10]:
# §5 — inventory, hashes, collision asserts.
QUEUES = [
    MINING_DIR / "candidates_target_receipts.yaml",
    MINING_DIR / "candidates_target_nullp.yaml",
    MINING_DIR / "candidates_target_random.yaml",
]
_all_cases = [*load_cases(), *load_target_cases()]
_case_texts = {c.text.strip().lower() for c in _all_cases}
_case_ids = {c.comment_id for c in _all_cases if c.comment_id}
_queue_ids: set[str] = set()

for q in QUEUES:
    payload = yaml.safe_load(q.read_text())["candidates"]
    digest = hashlib.sha256(q.read_bytes()).hexdigest()[:16]
    print(f"{q.name:36} {len(payload):3} candidates  sha256:{digest}")
    for c in payload:
        assert c["text"].strip().lower() not in _case_texts, f"{c['id']} collides with an existing case text"
        assert c["comment_id"] not in _case_ids, f"{c['id']} comment_id already used in a cases file"
        assert c["comment_id"] not in _queue_ids, f"{c['id']} duplicated across queues"
        _queue_ids.add(c["comment_id"])
print(f"\n{len(_queue_ids)} unique candidates; no collisions with tests/eval/cases.yaml or target_cases.yaml.")

candidates_target_receipts.yaml       70 candidates  sha256:8d77637dcc3fd642
candidates_target_nullp.yaml          35 candidates  sha256:7635c17f0700f700
candidates_target_random.yaml         30 candidates  sha256:70c6c69e8824c0b0

135 unique candidates; no collisions with tests/eval/cases.yaml or target_cases.yaml.


### Handoff checklist

1. **Label** each `candidates_target_*.yaml` (instructions in each file header):
   set `proposed_target` / `proposed_category`, rename to `expected_target` /
   `category`, delete rejects and `context:` blocks.
2. **Verify** with the nba-superfan agent (Claude Code, outside this notebook):
   a pass over the labeled queues for mislabeled or ambiguous entries and for
   alias-identity false positives that should be rejects.
3. **Merge** survivors into `tests/eval/target_cases.yaml`; every used category
   needs a `meta.category_floors` entry (placeholder `0.0`).
4. **Baseline**: `uv run pytest -m eval tests/eval/test_target_eval.py --maxfail=0 -rxX`
   × 3 runs against `TARGET_PROMPT_VERSION`; mark stable misses
   `known_miss: true`; pin floors (baseline minus one case of slack for
   categories with ≥ 5 cases).
5. `uv run pytest` (offline) to confirm nothing else moved.

**No prompt change happens in this notebook or its PR.** The toward-prompt
experiments and freeze are the next PR's job, measured against the suite this
feeds.